In [ ]:
import requests
import base64

url1= "https://api.github.com/repos/zju3dv/pvnet/contents/run.py?ref=master"
response = requests.get(url1)

data = response.json()
encoded_content = data['content']
decoded_content = base64.b64decode(encoded_content).decode("utf-8")
print(decoded_content)

In [ ]:
import requests
import base64

url1= "https://api.github.com/repos/zju3dv/pvnet/contents/"
response = requests.get(url1)

data=response.json()
print(data)
for content in data:
    if content=='content':
        print(True)
    print(content)


In [ ]:
from pydantic import BaseModel, Field
from typing import Optional
import base64
from agents import Agent, Runner, trace, OpenAIChatCompletionsModel, function_tool
import os

github_access_token=os.getenv('Github_access_token')
HEADERS = {"Authorization": github_access_token}

class Navigate_repo_class(BaseModel):
    list_url: Optional[list[dict]]
    content : Optional[str]

@function_tool
def Navigate_repo(url:str, file_type : str)-> Navigate_repo_class:
    '''
    When to call this function->
    One you have to navigate a tree or to get the content of blob

    What to pass in this function->
    1. url : GitHub API contents URL only. Must follow this format:
         https://api.github.com/repos/{user}/{repo}/contents/{path}
         
         NOT the HTML url (https://github.com/...) 
         Use URLs returned by return_file_structure or previous Navigate_repo calls.
         Never construct URLs manually.

    2. file_type : the type of the file the url is pointing towards (blob/tree)
    
    What does the function return->
    list_url : list of the child URLs present within the given url
    content : if the url file_type is a blob then return the content

    What to do next after calling this function->
    file_type='tree' → make parallel Navigate_repo calls on each URL in list_url
    file_type='blob' → pass content to the Parent agent, continue remaining trees
    '''


    response=requests.get(url, headers=HEADERS)
    Data=response.json()
    
    content=None
    response_list=[]

    if file_type=='tree':
        for data in Data:
            response_list.append({'name' : data.get('name'), 'URL' : data.get('url')})

    if file_type=='blob':
        content=base64.b64decode(Data['content']).decode("utf-8")

    return Navigate_repo_class(list_url=response_list, content=content)

@function_tool
def return_file_structure(user: str, repository_name:str):
    '''
    When to call this function->
    After reading the Readme of the repository to understand the repository structure

    What to pass in this function->
    1. user : owner of the repository
    2. repository_name : repository name as given on GitHub
    
    What does the function return->
    returns the string of the repository structure e.g. (Readme.md Blob \n asset tree)

    What to do next after calling this function->
    Use the tool Navigate_repo to explore and understand the repo
    '''

    repo_url=f'https://api.github.com/repos/{user}/{repository_name}/git/trees/master?recursive=1'
    
    response=requests.get(repo_url, headers=HEADERS)
    data=response.json() 

    structure_string=''

    for item in data["tree"]:
        structure_string+=f'''{item['path']}  {item['type']} \n'''

    return structure_string

@function_tool
def get_readme(user:str, repository_name:str):
    '''
    When to call->
    Always call this FIRST before any other tool.

    What to pass in->
    1. user : GitHub username or org name e.g. (https://github.com/zju3dv/pvnet here name is zju3dv)
    2. repository_name : exact repository name as on GitHub e.g. (https://github.com/zju3dv/pvnet here repository_name is pvnet)

    What does it return->
    str : full decoded README content, or 'Readme not found' if absent.

    What to do next->
    Call return_file_structure to map the full repo, using README 
    as context for which files matter.    
    '''

    url=f'''https://api.github.com/repos/{user}/{repository_name}/contents/'''

    response=requests.get(url, headers=HEADERS)

    Data=response.json()

    content='Readme not found'
    for data in Data:
        readme_check=data['name'].lower()
        if readme_check=='readme.md':
            url=data['url']
            result=requests.get(url, headers=HEADERS)
            result=result.json()
            content=result['content']
            content=base64.b64decode(content).decode("utf-8")

    return content

In [ ]:

parent_agent_instruction='''
You are a technical educator who creates in-depth tutorials for GitHub repositories.
You have three tools available to explore a repository. You MUST follow this exact 
workflow — do not skip any step.

═══════════════════════════════════════
STEP 1 — Read the README (MANDATORY FIRST STEP)
═══════════════════════════════════════
Call get_readme() first, always.
- Understand the project's purpose, architecture, and key components.
- Note every major feature, library, and module mentioned.
- This is your reference map for all subsequent decisions.

═══════════════════════════════════════
STEP 2 — Get the full file structure (MANDATORY SECOND STEP)
═══════════════════════════════════════
Call return_file_structure() immediately after reading the README.
- Study every path and identify which trees and blobs are relevant.
- Mark ALL files that likely contain core logic based on the README.
- Do NOT proceed to Step 3 until you have a clear picture of what to explore.

═══════════════════════════════════════
STEP 3 — Navigate and read ALL relevant files (MANDATORY — DO NOT SKIP)
═══════════════════════════════════════
You MUST call Navigate_repo() to read file contents. This is not optional.
Skipping this step means you have no actual code to reference — your tutorial 
will be incomplete and unacceptable.

Follow this process:
a) For every relevant directory (type=tree) → call Navigate_repo(url, 'tree')
   to get its children. You can call multiple trees in parallel.

b) For every relevant file (type=blob) → call Navigate_repo(url, 'blob')
   to get its content. You can call multiple blobs in parallel.

c) Prioritize files in this order:
   1. Entry point files (main.py, app.py, index.py, run.py etc.)
   2. Core logic files referenced in the README
   3. Configuration files (requirements.txt, config.py, .env.example)
   4. Utility/helper files that support the core logic
   5. Skip: LICENSE, .gitignore, __pycache__, test files, migration files

d) After reading each file, check for imports or references to other files
   you haven't read yet — navigate those too.

e) Read at minimum every file that contributes to the core functionality.
   Do not stop after 1-2 files.

YOU MUST NOT MOVE TO STEP 4 UNTIL YOU HAVE CALLED Navigate_repo() 
AT LEAST ONCE AND READ THE CORE FILES.

═══════════════════════════════════════
STEP 4 — Write the tutorial (ONLY after Step 3 is complete)
═══════════════════════════════════════
Write a tutorial of AT LEAST 2000 words following this EXACT section order:

─────────────────────────────────────
SECTION 1 — OVERVIEW
─────────────────────────────────────
- What problem does this project solve?
- Who is it for?
- What is the high-level approach — how does it solve the problem?
- What libraries are used and WHY was each chosen over alternatives?
- Keep this engaging — a reader should immediately understand the value
  of this project after reading this section.

─────────────────────────────────────
SECTION 2 — REPOSITORY STRUCTURE
─────────────────────────────────────
- Show the directory tree (use the output of return_file_structure)
- Annotate every file and folder with a one-line description of its role:

  ```
  repo/
  ├── main.py          # entry point — parses args and launches the pipeline
  ├── model/
  │   ├── network.py   # defines the neural network architecture
  │   └── loss.py      # custom loss functions
  ├── utils/
  │   └── helpers.py   # shared utility functions
  └── requirements.txt # project dependencies
  ```
- Do NOT just list file names — every line must have a purpose annotation.

─────────────────────────────────────
SECTION 3 — INSTALLATION
─────────────────────────────────────
- Prerequisites (Python version, CUDA, OS requirements if any)
- Step-by-step install commands from requirements.txt or setup.py
- Environment variables required with explanation of what each does
- Common install pitfalls and how to avoid them

─────────────────────────────────────
SECTION 4 — RUNNING THE PROJECT
─────────────────────────────────────
- The exact command(s) to run the project
- What each CLI argument or config option does
- What the expected output looks like
- Any modes (train vs test, dev vs prod) and how to switch between them

─────────────────────────────────────
SECTION 5 — CODE ARCHITECTURE
─────────────────────────────────────
- How the codebase is organized at a high level
- The design patterns used (e.g. pipeline, agent loop, MVC)
- How data flows through the system from input to output:

  User Input → Module A → Module B → Module C → Final Output

- Why the code is structured this way — what problem does this 
  architecture solve?

─────────────────────────────────────
SECTION 6 — KEY FILES EXPLANATION
─────────────────────────────────────
This is the most detailed section. For every core file you read:

  ### filename.py
  **Purpose:** one sentence on what this file does

  Walk through each key function in the file:

  ### function_name()
  ```python
  # actual code snippet from the file — do not invent or paraphrase code
  ```
  - What this function does
  - Why it is implemented this way
  - How it connects to other parts of the project
  - Any non-obvious logic that needs explanation

Repeat for every core file. Do not skip files you read in Step 3.

─────────────────────────────────────
SECTION 7 — EXAMPLE WORKFLOW
─────────────────────────────────────
- Walk through one complete, concrete end-to-end example
- Use a real or realistic input (not "foo/bar" placeholders)
- Trace exactly what happens at each stage of the code:
  "When you run X, function Y is called, which does Z, then passes 
   the result to function W, which..."
- Show the intermediate states and the final output
- This section should make the reader feel like they watched the 
  code run in front of them

─────────────────────────────────────
SECTION 8 — CUSTOMIZATION
─────────────────────────────────────
- What can be changed without breaking the project
- Config values, hyperparameters, or flags the user can tweak
- How to swap components (e.g. replace the model, change the dataset)
- How to extend the project with new features
- Any hooks or extension points the author built in

─────────────────────────────────────
SECTION 9 — TROUBLESHOOTING
─────────────────────────────────────
- The most common errors a user will hit and exactly how to fix them
- Focus on errors tied to actual code you read — not generic advice
- Format each as:

  **Problem:** description of the error or symptom
  **Cause:** why it happens (reference the relevant code)
  **Fix:** exact steps to resolve it

QUALITY REQUIREMENTS:
- Follow the section order above exactly — do not reorder or merge sections
- Every claim about the code MUST reference actual code read via Navigate_repo
- Minimum 2000 words
- At least 5 actual code snippets from the repository — never invent code
- No guessing — if you did not read a file, do not reference it
- Write for an intermediate developer — technical but clear
'''

In [ ]:
import os
import openai
from openai import AsyncOpenAI
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent

In [ ]:
from pathlib import Path
from Function_tools import Navigate_repo, get_readme, return_file_structure, create_chunks
from rich.console import Console
from agents import Agent, Runner, trace, OpenAIChatCompletionsModel, function_tool
from dotenv import load_dotenv
from Parent_agent import parent_agent_instruction
import os
console=Console()
load_dotenv()
client = AsyncOpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

model = OpenAIChatCompletionsModel(
    model="gpt-4o-mini",
    openai_client=client)
tools=[get_readme, return_file_structure, Navigate_repo, create_chunks]
Parent_Agent=Agent(name='Parent Agent', instructions=parent_agent_instruction, tools=tools, model=model)

message='explain the master branch of this repo https://github.com/soumil2334/Youtube-Ask-Feature '
repo_text=[]
with trace('GitHub Repo Explainer'):
    result = Runner.run_streamed(starting_agent=Parent_Agent, input=message)
    async for event in result.stream_events():
        if event.type=='raw_response_event' and isinstance(event.data, ResponseTextDeltaEvent):
            print(event.data.delta, end='')
            repo_text.append(event.data.delta)

REPO=Path('repo_text')
REPO.mkdir(parents=True, exist_ok=True)
text_file=REPO/'text1.txt'
repo=''.join(repo_text)

from save_as_pdf import save_as_pdf

save_as_pdf(repo, 'Repo_pdf.pdf')


In [ ]:
with open('repo.txt', 'w', encoding='utf-8') as f:
    f.write(repo)


In [ ]:
from rich.markdown import Markdown
console.print(Markdown(repo))

In [ ]:
with open('repo.txt', 'r', encoding='utf-8') as f:
    a=f.read()

In [ ]:
code_string='''import pdfkit
import markdown as md_converter
from pathlib import Path

def save_as_pdf(tutorial_text: str, filename: Path):
    # Convert markdown to HTML
    html_content = md_converter.markdown(
        tutorial_text,
        extensions=['fenced_code', 'tables', 'codehilite']
    )
    
    styled_html = f"""
    <html>
    <head>
        <style>
            body {{ font-family: Arial, sans-serif; margin: 40px; line-height: 1.6; }}
            code {{ background: #f4f4f4; padding: 2px 6px; border-radius: 3px; }}
            pre  {{ background: #f4f4f4; padding: 15px; border-radius: 5px; }}
            h1, h2, h3 {{ color: #2c3e50; }}
        </style>
    </head>
    <body>{html_content}</body>
    </html>
    """

    # Point to wkhtmltopdf install location
    config = pdfkit.configuration(
        wkhtmltopdf=r'C:\Program Files\wkhtmltopdf\bin\wkhtmltopdf.exe'
    )
    
    save_path=Path(filename)
    save_path.mkdir(parents=True, exist_ok=True)
    repo_pdf=save_path/'repo.pdf'

    pdfkit.from_string(
    styled_html, 
    repo_pdf, 
    configuration=config,
    options={
        'encoding': 'UTF-8',
        'enable-local-file-access': None
    }
)
    print(f"Saved to {save_path}")'''

In [ ]:
from tree_sitter_languages import get_parser
def get_tree(code_string: str, language: str):
    parser = get_parser(language)
    tree = parser.parse(bytes(code_string, "utf8"))
    return tree

tree = get_tree(code_string, "python")

In [ ]:
print(tree.root_node)

In [ ]:
root = tree.root_node

def print_tree(node, code_string, indent=0):
    text = code_string[node.start_byte:node.end_byte]
    short_text = text[:40].replace('\n', '\\n')
    print(" " * indent + f"[{node.type}] → '{short_text}'")
    for child in node.children:
        print_tree(child, code_string, indent + 2)

print_tree(root, code_string)

In [ ]:
from Function_tools import return_file_structure
return_file_structure(user="soumil2334", repository_name="Youtube-Ask-Feature", branch='master')

In [2]:
graph_docs

[[GraphDocument(nodes=[Node(id='Upload_Endpoint', type='Endpoint', properties={}), Node(id='Job_Id', type='Parameter', properties={}), Node(id='Audio_Path', type='File', properties={}), Node(id='Result', type='Dictionary', properties={}), Node(id='Segments', type='List', properties={}), Node(id='Parent_Dir', type='Directory', properties={}), Node(id='Text_Path', type='File', properties={}), Node(id='Collection_Name', type='String', properties={}), Node(id='Chromadb_Dir', type='Directory', properties={}), Node(id='Exception', type='Exception', properties={})], relationships=[Relationship(source=Node(id='Upload_Endpoint', type='Endpoint', properties={}), target=Node(id='Job_Id', type='Parameter', properties={}), type='POST', properties={}), Relationship(source=Node(id='Job_Id', type='Parameter', properties={}), target=Node(id='Audio_Path', type='File', properties={}), type='GENERATES', properties={}), Relationship(source=Node(id='Audio_Path', type='File', properties={}), target=Node(id='

In [11]:
graph_documents

[GraphDocument(nodes=[Node(id='Upload_Endpoint', type='Endpoint', properties={}), Node(id='Job_Id', type='Parameter', properties={}), Node(id='Audio_Path', type='File', properties={}), Node(id='Result', type='Dictionary', properties={}), Node(id='Segments', type='List', properties={}), Node(id='Parent_Dir', type='Directory', properties={}), Node(id='Text_Path', type='File', properties={}), Node(id='Collection_Name', type='String', properties={}), Node(id='Chromadb_Dir', type='Directory', properties={}), Node(id='Exception', type='Exception', properties={})], relationships=[Relationship(source=Node(id='Upload_Endpoint', type='Endpoint', properties={}), target=Node(id='Job_Id', type='Parameter', properties={}), type='POST', properties={}), Relationship(source=Node(id='Job_Id', type='Parameter', properties={}), target=Node(id='Audio_Path', type='File', properties={}), type='GENERATES', properties={}), Relationship(source=Node(id='Audio_Path', type='File', properties={}), target=Node(id='R

In [ ]:
len(graph_documents)

41

In [31]:
c=graph_documents[1].source
print(c)
extra_string=''
for key, value in c.metadata.items():
    extra_string+=f'{key.upper()} : {value}\n'

extra_string+=f'PAGE_CONTENT : {c.page_content}'
print(extra_string)

page_content='logging.basicConfig(
    level=logging.ERROR,
    format="%(asctime)s | %(levelname)s | %(filename)s:%(lineno)d | %(funcName)s() | %(message)s"
)' metadata={'file': 'graph/graph_nodes.py', 'node_type': 'expression_statement', 'name': 'expression_statement', 'start_line': 13, 'end_line': 16, 'id': '4333fa5a4f60a0053dd02cabceb259be'}
FILE : graph/graph_nodes.py
NODE_TYPE : expression_statement
NAME : expression_statement
START_LINE : 13
END_LINE : 16
ID : 4333fa5a4f60a0053dd02cabceb259be
PAGE_CONTENT : logging.basicConfig(
    level=logging.ERROR,
    format="%(asctime)s | %(levelname)s | %(filename)s:%(lineno)d | %(funcName)s() | %(message)s"
)


In [ ]:
## Creating a list containing the text and payload
def create_string_payload(graph_docs:list):
    
    lists=[]
    for graph_doc in graph_docs:
        payload={}
        temp_string=''
        source=graph_doc.source

        for key,value in source.metadata.items():
            payload[key]=value
            temp_string+=f'{key.upper()} : {value}\n'

        temp_string+=f'CODE : {source.page_content}\n'
        payload['Code']=source.page_content

        nodes=graph_doc.nodes
        nodes_list=[]
        for node in nodes:
            nodes_list.append({'Node':node.id, 'type':node.type})
         
        temp_string+=f'NODES : {nodes_list}'
        payload['Nodes']=nodes_list

        relationships_list=[]
        relationships=graph_doc.relationships
        for relationship in relationships:
            relationships_list.append(f'{relationship.source.id} {relationship.type} {relationship.target.id}')

        
        temp_string+=f'NODES_RELATIONSHIP : {relationships_list}'   
        payload["Node_Relationship"]=relationships_list

        lists.append({'TEXT': temp_string, 'PAYLOAD' : payload})
    
    return lists

In [32]:
b=graph_documents[1].nodes
nodes=[]
print(b)
for B in b:
    nodes.append({'Node':B.id, 'type':B.type})
print(nodes, end='\n')

[Node(id='Logging', type='Module', properties={}), Node(id='Basicconfig', type='Function', properties={}), Node(id='Error', type='Loglevel', properties={}), Node(id='Format', type='String', properties={})]
[{'Node': 'Logging', 'type': 'Module'}, {'Node': 'Basicconfig', 'type': 'Function'}, {'Node': 'Error', 'type': 'Loglevel'}, {'Node': 'Format', 'type': 'String'}]


In [34]:
a=graph_documents[1].relationships
print(len(a))
print(a)
print(f'{a[0].source.id} {a[0].type} {a[0].target.id}')

3
[Relationship(source=Node(id='Logging', type='Module', properties={}), target=Node(id='Basicconfig', type='Function', properties={}), type='CONTAINS', properties={}), Relationship(source=Node(id='Basicconfig', type='Function', properties={}), target=Node(id='Error', type='Loglevel', properties={}), type='SET_LEVEL', properties={}), Relationship(source=Node(id='Basicconfig', type='Function', properties={}), target=Node(id='Format', type='String', properties={}), type='SET_FORMAT', properties={})]
Logging CONTAINS Basicconfig


In [14]:
from langchain_neo4j import Neo4jGraph
import os
from dotenv import load_dotenv
load_dotenv()

graph=Neo4jGraph(
    url='neo4j+s://5ea5a11a.databases.neo4j.io',
    username='5ea5a11a',
    password='znlpXCHgBWD5TGkUt7sGJ4M0_dabGf79sj-8xh2Z9k'
)
list_of_docs=[]
graph.add_graph_documents(
        graph_documents=graph_documents,
        baseEntityLabel=True,
        include_source=True
    )

ValueError: Could not connect to Neo4j database. Please ensure that the credentials are correct

In [ ]:
from langchain_neo4j import Neo4jGraph, GraphCypherQAChain
from langchain_openai import ChatOpenAI
graph=Neo4jGraph(
    url='neo4j+s://5ea5a11a.databases.neo4j.io',
    username='5ea5a11a',
    password='AznlpXCHgBWD5TGkUt7sGJ4M0_dabGf79sj-8xh2Z9k'
)
graph.refresh_schema()  # important — lets LLM know your graph structure

llm = ChatOpenAI(model="gpt-4o", temperature=0)

chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    verbose=True,
    allow_dangerous_requests=True
)

result = chain.invoke({"query": "What functions are defined in main.py?"})
print(result["result"])